<a href="https://colab.research.google.com/github/tlsgptj/2024-Samsung-AI-Challenge-Black-box-Optimization/blob/main/2025_Samsung_Collegiate_Programming_Challenge_AI_%EC%B1%8C%EB%A6%B0%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p input_images

In [ ]:
!mv *.jpg input_images/

In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

In [ ]:
# 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

In [ ]:
# 모델 로딩
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)

In [ ]:
# 정답 알파벳 추출 함수
def extract_answer_letter(text):
    match = re.search(r"Answer:\s*([A-Da-d])\b", text)
    return match.group(1).upper() if match else "?"

In [ ]:
test = pd.read_csv('./dev_test.csv')
results = []

for _, row in tqdm(test.iterrows(), total=len(test)):
    image = Image.open(row['img_path']).convert("RGB")
    choices = [row[c] for c in ['A', 'B', 'C', 'D']]

    prompt = (
        "You are a helpful AI that answers multiple-choice questions based on the given image.\n"
        "Select the best answer from A, B, C, or D.\n\n"
        f"Question: {row['Question']}\n"
        + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]) +
        "\nAnswer:"
    )

    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: (v.half().to(device) if v.dtype == torch.float32 else v.to(device)) for k, v in inputs.items()}

    output = model.generate(**inputs, max_new_tokens=3, do_sample=False, temperature=0.0)
    decoded = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()
    results.append(extract_answer_letter(decoded))
print('✅ Done.')

In [ ]:
submission = pd.read_csv('./sample_submission.csv')
submission['answer'] = results
submission.to_csv('./baseline_submit.csv', index=False)
print("✅ Done.")

In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# ✅ 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

# ✅ 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# ✅ 모델 로딩
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)

# ✅ 정답 추출 함수
def extract_answer_letter(text):
    match = re.search(r"([A-D])(\.|[\s])", text)
    if match:
        return match.group(1).upper()
    for c in "ABCD":
        if c in text:
            return c
    return "?"

# ✅ 데이터 로드
test_df = pd.read_csv("./dev_test.csv")  # 컬럼: ID, img_path, Question, A, B, C, D
results = []

# ✅ 추론
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        image = Image.open(row["img_path"]).convert("RGB")
        choices = [row["A"], row["B"], row["C"], row["D"]]

        prompt = (
            "You are a helpful AI assistant that answers multiple-choice questions based on an image.\n"
            "Choose the most appropriate answer from A, B, C, or D.\n\n"
            f"Question: {row['Question']}\n"
            + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)])
            + "\nThe correct answer is:"
        )

        inputs = processor(images=image, text=prompt, return_tensors="pt")
        inputs = {k: (v.half().to(device) if v.dtype == torch.float32 else v.to(device))
                  for k, v in inputs.items()}

        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=True,
            temperature=0.3,
            top_p=0.9
        )
        decoded = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()
        answer = extract_answer_letter(decoded)

        results.append(answer)
    except Exception as e:
        print(f"[ERROR] {row['ID']} - {e}")
        results.append("?")

print("✅ Inference Done.")

# ✅ 제출 파일 저장
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "answer": results
})
submission.to_csv("improved_submit.csv", index=False)
print("✅ Saved as improved_submit.csv")

In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# 환경설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 시드 고정 (재현성)
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

# 모델 및 프로세서 로드 (BLIP2 base 모델 추천)
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)


# 정답 추출 함수 (A-D 중에서 추출)
def extract_answer_letter(text):
    match = re.search(r"([A-D])(\.|[\s])", text)
    if match:
        return match.group(1).upper()
    for c in "ABCD":
        if c in text:
            return c
    return "?"

# 데이터 전처리: 이미지 리사이즈 (모델 입력 크기 맞춤)
def preprocess_image(image_path, size=(384, 384)):
    img = Image.open(image_path).convert("RGB")
    img = img.resize(size, resample=Image.BILINEAR)
    return img

# 하이퍼파라미터 설정
temperature = 0.3
top_p = 0.9

# 데이터 로드
test_df = pd.read_csv("./dev_test.csv")  # ID, img_path, Question, A, B, C, D
results = []

# 추론 루프
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        image = preprocess_image(row["img_path"])  # 전처리 적용
        choices = [row["A"], row["B"], row["C"], row["D"]]

        prompt = (
            "You are a helpful AI assistant that answers multiple-choice questions based on an image.\n"
            "Choose the most appropriate answer from A, B, C, or D.\n\n"
            f"Question: {row['Question']}\n"
            + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)])
            + "\nAnswer:"
        )

        inputs = processor(images=image, text=prompt, return_tensors="pt")
        inputs = {k: (v.half().to(device) if v.dtype == torch.float32 else v.to(device))
                  for k, v in inputs.items()}

        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=1,
        )
        decoded = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()
        answer = extract_answer_letter(decoded)

        # 후처리 규칙 예: 답변이 ?일 경우 가장 많이 등장하는 선택지로 대체 가능 (예시)
        if answer == "?":
            answer = max(set(choices), key=choices.count)

        results.append(answer)

    except Exception as e:
        print(f"[ERROR] {row['ID']} - {e}")
        results.append("?")

print("Inference Done.")

# 제출 저장
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "answer": results
})
submission.to_csv("submission.csv", index=False)
print("Saved as submission.csv")

In [ ]:
# ⚙️ 1. 라이브러리 설치 (최초 1회만 실행)
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate

# ✅ import
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration

# ✅ 환경 설정
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

# ✅ 모델 로딩
model_id = "llava-hf/llava-1.5-7b-hf"
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# ✅ 정답 추출
def extract_answer_letter(text):
    match = re.search(r"\b([A-D])[\.:]?\b", text)
    if match:
        return match.group(1).upper()
    for c in "ABCD":
        if c in text.upper():
            return c
    return "?"

# ✅ 이미지 전처리
def preprocess_image(image_path, size=(336, 336)):
    img = Image.open(image_path).convert("RGB")
    return img.resize(size)

# ✅ CSV 로드
csv_path = "/content/dev_test.csv"  # 경로 맞게 수정
df = pd.read_csv(csv_path)
results = []

# ✅ 추론
for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        # 이미지 로딩
        image_path = os.path.join("/content", row["img_path"].lstrip("./"))
        image = preprocess_image(image_path)

        choices = [row["A"], row["B"], row["C"], row["D"]]
        prompt = (
            "You are a helpful AI assistant that answers multiple-choice questions based on an image.\n"
            "Choose the most appropriate answer from A, B, C, or D.\n\n"
            f"Question: {row['Question']}\n" +
            "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]) +
            "\nAnswer:"
        )

        # 🔥 텍스트/이미지 따로 처리
        text_inputs = processor.tokenizer(prompt, return_tensors="pt").to(device)
        image_inputs = processor.image_processor(image, return_tensors="pt").to(device)
        image_tensor = image_inputs["pixel_values"]

        # 🧠 모델 추론
        output = model.generate(
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"],
            images=image_tensor,
            max_new_tokens=10,
            do_sample=False
        )

        decoded = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()
        answer = extract_answer_letter(decoded)

        if answer == "?":
            answer = "D"  # 후처리 기본값 설정

        results.append(answer)

    except Exception as e:
        print(f"[ERROR] {row['ID']} - {e}")
        results.append("?")

# ✅ 결과 저장
submission = pd.DataFrame({
    "ID": df["ID"],
    "answer": results
})
submission.to_csv("/content/submission_llava.csv", index=False)
print("✅ Saved: submission_llava.csv")


In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# ✅ 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

# ✅ 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# ✅ 모델 로딩
print("📥 Loading BLIP2 model...")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Model loaded successfully!")

# ✅ 정답 추출 함수
def extract_answer_letter(text):
    """생성된 텍스트에서 A, B, C, D 답안을 추출"""
    # 프롬프트 이후의 실제 생성된 부분만 추출
    if "The correct answer is:" in text:
        # "The correct answer is:" 이후 부분만 가져오기
        answer_part = text.split("The correct answer is:")[-1].strip()
    else:
        answer_part = text

    # 첫 번째 패턴: "A", "B", "C", "D" (단독으로 나타나는 경우)
    match = re.search(r'\b([A-D])\b', answer_part)
    if match:
        return match.group(1).upper()

    # 두 번째 패턴: "A.", "B.", "C.", "D."
    match = re.search(r'([A-D])\.', answer_part)
    if match:
        return match.group(1).upper()

    # 세 번째 패턴: "Answer: A", "Answer is A" 등
    match = re.search(r'(?:answer|Answer)(?:\s*:?\s*is?)?\s*([A-D])', answer_part)
    if match:
        return match.group(1).upper()

    # 마지막 패턴: 첫 번째로 나오는 A, B, C, D
    for c in "ABCD":
        if c in answer_part:
            return c

    return "?"

# ✅ 데이터 로드 (코랩 환경)
data_path = "/content/dev_test.csv"  # 코랩 기본 경로
if not os.path.exists(data_path):
    # 다른 가능한 경로들 확인
    possible_paths = [
        "./dev_test.csv",
        "/content/drive/MyDrive/dev_test.csv",  # 구글 드라이브 연동시
        "../dev_test.csv",
        "./dataset/dev_test.csv",
        "/content/dataset/dev_test.csv"
    ]

    for path in possible_paths:
        if os.path.exists(path):
            data_path = path
            break
    else:
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다. 다음 경로들을 확인해주세요: {possible_paths}")

print(f"📊 Loading data from: {data_path}")
test_df = pd.read_csv(data_path)
print(f"✅ Loaded {len(test_df)} samples")

# 이미지 경로 검증 및 수정 (코랩 환경)
def validate_image_path(img_path):
    """이미지 경로를 검증하고 수정 (코랩 환경)"""
    # 코랩 절대 경로로 변환
    colab_path = img_path.replace("./input_images/", "/content/input_images/")

    if os.path.exists(colab_path):
        return colab_path

    # 다른 가능한 경로들 시도
    possible_paths = [
        img_path,  # 원본 경로
        os.path.join("/content", img_path.lstrip("./")),  # 상대경로를 절대경로로
        os.path.join("/content/input_images", os.path.basename(img_path)),
        os.path.join("/content/data", os.path.basename(img_path)),
        os.path.join("/content/dataset", os.path.basename(img_path)),
        os.path.join("/content/drive/MyDrive/input_images", os.path.basename(img_path))  # 구글 드라이브
    ]

    for path in possible_paths:
        if os.path.exists(path):
            return path

    return None

# ✅ 추론 시작
print("🚀 Starting inference...")
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    try:
        # 이미지 경로 검증
        img_path = validate_image_path(row["img_path"])
        if img_path is None:
            print(f"[WARNING] Image not found: {row['img_path']}")
            results.append("?")
            continue

        # 이미지 로드
        image = Image.open(img_path).convert("RGB")

        # 선택지 준비
        choices = [row["A"], row["B"], row["C"], row["D"]]

        # 프롬프트 생성
        prompt = (
            "You are a helpful AI assistant that answers multiple-choice questions based on an image.\n"
            "Choose the most appropriate answer from A, B, C, or D.\n\n"
            f"Question: {row['Question']}\n"
            + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)])
            + "\nThe correct answer is:"
        )

        # 모델 입력 준비
        inputs = processor(images=image, text=prompt, return_tensors="pt")
        inputs = {k: (v.half().to(device) if v.dtype == torch.float32 else v.to(device))
                  for k, v in inputs.items()}

        # 추론 실행
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=10,  # 토큰 수 늘림
                do_sample=False,    # 결정적 생성
                temperature=1.0,    # 온도 조정
                pad_token_id=processor.tokenizer.eos_token_id
            )

        # 결과 디코딩 및 답안 추출 (프롬프트 제거)
        input_length = inputs['input_ids'].shape[1]
        generated_tokens = output[0][input_length:]  # 새로 생성된 부분만
        decoded = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        answer = extract_answer_letter(decoded)
        results.append(answer)

        # 진행상황 출력 (일부 샘플만)
        if idx < 5 or idx % 100 == 0:
            print(f"[Sample {idx}] Question: {row['Question'][:50]}...")
            print(f"[Sample {idx}] Generated (new tokens only): {decoded}")
            print(f"[Sample {idx}] Answer: {answer}")
            print("-" * 50)

    except Exception as e:
        print(f"[ERROR] Sample {idx} (ID: {row.get('ID', 'Unknown')}) - {e}")
        results.append("?")

print("✅ Inference completed!")

# ✅ 결과 통계
answer_counts = pd.Series(results).value_counts()
print("\n📊 Answer distribution:")
for answer, count in answer_counts.items():
    print(f"{answer}: {count} ({count/len(results)*100:.1f}%)")

# ✅ 제출 파일 저장 (코랩 환경)
output_dir = "/content/output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "answer": results
})

output_path = os.path.join(output_dir, "improved_submit.csv")
submission.to_csv(output_path, index=False)
print(f"✅ Results saved to: {output_path}")

# 백업 파일도 저장 (코랩 기본 디렉토리)
backup_path = "/content/improved_submit.csv"
submission.to_csv(backup_path, index=False)
print(f"✅ Backup saved to: {backup_path}")

# 구글 드라이브 연동시 추가 저장 (선택사항)
drive_path = "/content/drive/MyDrive/improved_submit.csv"
try:
    if os.path.exists("/content/drive/MyDrive"):
        submission.to_csv(drive_path, index=False)
        print(f"✅ Google Drive backup saved to: {drive_path}")
except Exception as e:
    print(f"[INFO] Google Drive backup failed (드라이브가 마운트되지 않았을 수 있습니다): {e}")

print(f"\n🎉 Process completed! Total samples: {len(results)}")
print(f"📄 Main submission file: {output_path}")
print(f"📄 Backup file: {backup_path}")

In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# ✅ 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

# ✅ 시드 고정
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# ✅ 모델 로딩
print("📥 Loading BLIP2 model...")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Model loaded successfully!")

# ✅ 개선된 정답 추출 함수
def extract_answer_letter(text, choices_text=""):
    """생성된 텍스트에서 A, B, C, D 답안을 추출 (개선된 버전)"""
    # 텍스트 정리
    text = text.strip()

    # 패턴들을 우선순위 순으로 정렬
    patterns = [
        # 1. 명확한 답변 패턴
        r'(?:answer|Answer|ANSWER)(?:\s*:?\s*is?)?\s*([A-D])',
        r'(?:correct|Correct|CORRECT)(?:\s*:?\s*is?)?\s*([A-D])',
        r'(?:choice|Choice|option|Option)(?:\s*:?\s*is?)?\s*([A-D])',

        # 2. 괄호나 따옴표로 둘러싸인 답변
        r'[\(\[\"\']([A-D])[\)\]\"\']',

        # 3. 점이나 콜론 뒤의 답변
        r'([A-D])[\.:]',

        # 4. 단독으로 나타나는 대문자 (단어 경계 사용)
        r'\b([A-D])\b',

        # 5. 문장 시작의 대문자
        r'^([A-D])',

        # 6. 선택지 내용 매칭 (부분 매칭)
        r'([A-D])[\.\s]'
    ]

    # 각 패턴을 순서대로 시도
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
        if matches:
            # 첫 번째 매치를 대문자로 반환
            candidate = matches[0].upper()
            if candidate in "ABCD":
                return candidate

    # 마지막 시도: 텍스트에서 A, B, C, D 순서대로 찾기
    for letter in "ABCD":
        if letter in text.upper():
            return letter

    return "?"

# ✅ 개선된 프롬프트 생성 함수
def create_enhanced_prompt(question, choices):
    """더 효과적인 프롬프트 생성"""

    # 다양한 프롬프트 템플릿 중 하나를 선택 (randomization으로 다양성 확보)
    templates = [
        # Template 1: 단계별 추론 유도
        """Look at this image carefully and answer the multiple choice question.

Question: {question}

Options:
A. {choice_a}
B. {choice_b}
C. {choice_c}
D. {choice_d}

Think step by step and choose the best answer. The answer is:""",

        # Template 2: 직접적인 지시
        """Based on what you see in the image, select the correct answer.

{question}

A. {choice_a}
B. {choice_b}
C. {choice_c}
D. {choice_d}

Answer:""",

        # Template 3: 명확한 지시
        """Analyze the image and choose the most appropriate answer from the options below.

Question: {question}

A) {choice_a}
B) {choice_b}
C) {choice_c}
D) {choice_d}

The correct choice is:"""
    ]

    # 랜덤하게 템플릿 선택 (일정한 다양성 확보)
    template = random.choice(templates)

    return template.format(
        question=question,
        choice_a=choices[0],
        choice_b=choices[1],
        choice_c=choices[2],
        choice_d=choices[3]
    )

# ✅ 데이터 로드 (코랩 환경)
data_path = "/content/dev_test.csv"  # 코랩 기본 경로
if not os.path.exists(data_path):
    # 다른 가능한 경로들 확인
    possible_paths = [
        "./dev_test.csv",
        "/content/drive/MyDrive/dev_test.csv",  # 구글 드라이브 연동시
        "../dev_test.csv",
        "./dataset/dev_test.csv",
        "/content/dataset/dev_test.csv"
    ]

    for path in possible_paths:
        if os.path.exists(path):
            data_path = path
            break
    else:
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다. 다음 경로들을 확인해주세요: {possible_paths}")

print(f"📊 Loading data from: {data_path}")
test_df = pd.read_csv(data_path)
print(f"✅ Loaded {len(test_df)} samples")

# 이미지 경로 검증 및 수정 (코랩 환경)
def validate_image_path(img_path):
    """이미지 경로를 검증하고 수정 (코랩 환경)"""
    # 코랩 절대 경로로 변환
    colab_path = img_path.replace("./input_images/", "/content/input_images/")

    if os.path.exists(colab_path):
        return colab_path

    # 다른 가능한 경로들 시도
    possible_paths = [
        img_path,  # 원본 경로
        os.path.join("/content", img_path.lstrip("./")),  # 상대경로를 절대경로로
        os.path.join("/content/input_images", os.path.basename(img_path)),
        os.path.join("/content/data", os.path.basename(img_path)),
        os.path.join("/content/dataset", os.path.basename(img_path)),
        os.path.join("/content/drive/MyDrive/input_images", os.path.basename(img_path))  # 구글 드라이브
    ]

    for path in possible_paths:
        if os.path.exists(path):
            return path

    return None

# ✅ 다중 시도 추론 함수
def multi_attempt_inference(image, prompt, max_attempts=3):
    """여러 번 시도하여 더 안정적인 답변 얻기"""

    generation_params = [
        # 시도 1: 보수적 설정
        {
            "max_new_tokens": 20,
            "do_sample": True,
            "temperature": 0.3,
            "top_p": 0.8,
            "repetition_penalty": 1.1
        },
        # 시도 2: 중간 설정
        {
            "max_new_tokens": 15,
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.0
        },
        # 시도 3: 결정적 설정
        {
            "max_new_tokens": 10,
            "do_sample": False,
            "temperature": 1.0,
            "repetition_penalty": 1.0
        }
    ]

    for attempt in range(max_attempts):
        try:
            # 모델 입력 준비
            inputs = processor(images=image, text=prompt, return_tensors="pt")
            inputs = {k: (v.half().to(device) if v.dtype == torch.float32 else v.to(device))
                      for k, v in inputs.items()}

            # 현재 시도의 파라미터 사용
            params = generation_params[attempt]

            # 추론 실행
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    pad_token_id=processor.tokenizer.eos_token_id,
                    **params
                )

            # 결과 디코딩 (프롬프트 제거)
            input_length = inputs['input_ids'].shape[1]
            generated_tokens = output[0][input_length:]
            decoded = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

            # 답변 추출
            answer = extract_answer_letter(decoded)

            # 유효한 답변이면 반환
            if answer in "ABCD":
                return answer, decoded

        except Exception as e:
            print(f"[WARNING] Attempt {attempt + 1} failed: {e}")
            continue

    # 모든 시도 실패시 ? 반환
    return "?", ""

# ✅ 추론 시작
print("🚀 Starting inference...")
results = []
generation_logs = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    try:
        # 이미지 경로 검증
        img_path = validate_image_path(row["img_path"])
        if img_path is None:
            print(f"[WARNING] Image not found: {row['img_path']}")
            results.append("?")
            generation_logs.append("Image not found")
            continue

        # 이미지 로드
        image = Image.open(img_path).convert("RGB")

        # 선택지 준비
        choices = [row["A"], row["B"], row["C"], row["D"]]

        # 개선된 프롬프트 생성
        prompt = create_enhanced_prompt(row['Question'], choices)

        # 다중 시도 추론
        answer, generated_text = multi_attempt_inference(image, prompt)
        results.append(answer)
        generation_logs.append(generated_text)

        # 진행상황 출력 (일부 샘플만)
        if idx < 5 or idx % 20 == 0:
            print(f"[Sample {idx}] Question: {row['Question'][:50]}...")
            print(f"[Sample {idx}] Generated: {generated_text}")
            print(f"[Sample {idx}] Answer: {answer}")
            print("-" * 50)

    except Exception as e:
        print(f"[ERROR] Sample {idx} (ID: {row.get('ID', 'Unknown')}) - {e}")
        results.append("?")
        generation_logs.append(f"Error: {str(e)}")

print("✅ Inference completed!")

# ✅ 결과 통계
answer_counts = pd.Series(results).value_counts()
print("\n📊 Answer distribution:")
for answer, count in answer_counts.items():
    print(f"{answer}: {count} ({count/len(results)*100:.1f}%)")

# ✅ 결과 분석
print(f"\n📈 Analysis:")
print(f"Total samples: {len(results)}")
print(f"Valid answers (A-D): {sum(1 for r in results if r in 'ABCD')} ({sum(1 for r in results if r in 'ABCD')/len(results)*100:.1f}%)")
print(f"Failed extractions (?): {results.count('?')} ({results.count('?')/len(results)*100:.1f}%)")

# ✅ 제출 파일 저장 (코랩 환경)
output_dir = "/content/output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "answer": results
})

output_path = os.path.join(output_dir, "enhanced_submit.csv")
submission.to_csv(output_path, index=False)
print(f"✅ Results saved to: {output_path}")

# 백업 파일도 저장 (코랩 기본 디렉토리)
backup_path = "/content/enhanced_submit.csv"
submission.to_csv(backup_path, index=False)
print(f"✅ Backup saved to: {backup_path}")

# 디버깅용 상세 결과 저장
debug_df = pd.DataFrame({
    "ID": test_df["ID"],
    "Question": test_df["Question"],
    "answer": results,
    "generated_text": generation_logs
})
debug_path = "/content/debug_results.csv"
debug_df.to_csv(debug_path, index=False)
print(f"✅ Debug results saved to: {debug_path}")

# 구글 드라이브 연동시 추가 저장 (선택사항)
drive_path = "/content/drive/MyDrive/enhanced_submit.csv"
try:
    if os.path.exists("/content/drive/MyDrive"):
        submission.to_csv(drive_path, index=False)
        print(f"✅ Google Drive backup saved to: {drive_path}")
except Exception as e:
    print(f"[INFO] Google Drive backup failed (드라이브가 마운트되지 않았을 수 있습니다): {e}")

print(f"\n🎉 Process completed! Total samples: {len(results)}")
print(f"📄 Main submission file: {output_path}")
print(f"📄 Backup file: {backup_path}")
print(f"📄 Debug file: {debug_path}")

In [ ]:
!pip install transformers accelerate sentencepiece bitsandbytes

In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

# 3B이하 모델로 구성 필요
# 이미지가 옵셔널한경우 ->모델 하나 가능
#이미지 텍스트화 -> 모델 설명-> 답


print("Loading BLIP2 model...")
blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    device_map="auto",
    torch_dtype=torch.float16
)
print("BLIP2 loaded!")

llm_model_id = "Open-Orca/Mistral-7B-OpenOrca"
print(f"Loading LLM model: {llm_model_id}")
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)
llm_pipe = pipeline("text-generation", model=llm_model, tokenizer=llm_tokenizer)
print("LLM loaded!")

def extract_answer_letter(text):
    text = text.upper()

    # 1. "ANSWER: X" 또는 "ANSWER X" 형태 우선 탐색
    m = re.search(r'ANSWER[:\s]*([ABCD])', text)
    if m:
        return m.group(1)

    # 2. "CHOICE: X", "CORRECT: X", "OPTION: X" 탐색
    m = re.search(r'(CHOICE|CORRECT|OPTION)[:\s]*([ABCD])', text)
    if m:
        return m.group(2)

    # 3. 단독 A, B, C, D 첫 등장
    m = re.search(r'\b([ABCD])\b', text)
    if m:
        return m.group(1)

    return "?"

def validate_image_path(img_path):
    colab_path = img_path.replace("./input_images/", "/content/input_images/")
    if os.path.exists(colab_path):
        return colab_path
    for path in [
        img_path,
        os.path.join("/content", img_path.lstrip("./")),
        os.path.join("/content/input_images", os.path.basename(img_path)),
        os.path.join("/content/drive/MyDrive/input_images", os.path.basename(img_path))
    ]:
        if os.path.exists(path):
            return path
    return None

def generate_image_description(image):
    inputs = blip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        output = blip_model.generate(**inputs, max_new_tokens=50, num_beams=3)
        # num_beans=3~5 사이가 일반적, 너무 느려짐, overfitting 가능성 높음
    return blip_processor.tokenizer.decode(output[0], skip_special_tokens=True).strip()

def ask_local_llm(description, question, choices):
    prompt = f"""
You are given an image description and a multiple-choice question.
Read the description carefully and choose the best option based on the image's context.

Image Description: "{description}"

Question: {question}

Options:
A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Think carefully. Only respond with one letter (A, B, C, or D).

Answer:
"""

    output = llm_pipe(prompt, max_new_tokens=30, do_sample=True, temperature=0.7, top_p=0.9)[0]["generated_text"]

    # 'Answer:' 이후 내용만 처리
    if "Answer:" in output:
        output = output.split("Answer:")[-1].strip()

    answer = extract_answer_letter(output)
    return answer, output

data_path = "/content/dev_test.csv"
for alt_path in [
    data_path,
    "./dev_test.csv",
    "/content/drive/MyDrive/dev_test.csv",
    "./dataset/dev_test.csv"
]:
    if os.path.exists(alt_path):
        data_path = alt_path
        break
else:
    raise FileNotFoundError("CSV 파일을 찾을 수 없습니다.")

print(f"Loading data from: {data_path}")
test_df = pd.read_csv(data_path)
print(f"Loaded {len(test_df)} samples")

results, logs = [], []

print("Starting inference...")
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    try:
        img_path = validate_image_path(row["img_path"])
        if img_path is None:
            results.append("?")
            logs.append("Image not found")
            continue

        image = Image.open(img_path).convert("RGB")
        choices = [row["A"], row["B"], row["C"], row["D"]]

        desc = generate_image_description(image)
        answer, gen = ask_local_llm(desc, row["Question"], choices)

        results.append(answer)
        logs.append(gen)

        if idx < 5 or idx % 20 == 0:
            print(f"[{idx}] Image Desc: {desc}")
            print(f"[{idx}] Generated: {gen}")
            print(f"[{idx}] Answer: {answer}")
            print("-" * 50)

    except Exception as e:
        results.append("?")
        logs.append(f"Error: {e}")
        print(f"[ERROR {idx}] {e}")

output_dir = "/content/output"
os.makedirs(output_dir, exist_ok=True)

submission = pd.DataFrame({"ID": test_df["ID"], "answer": results})
submission.to_csv(os.path.join(output_dir, "final_submit.csv"), index=False)
submission.to_csv("/content/final_submit.csv", index=False)

pd.DataFrame({
    "ID": test_df["ID"],
    "Question": test_df["Question"],
    "answer": results,
    "generated_text": logs
}).to_csv("/content/debug_results.csv", index=False)

print("Inference completed!")


In [ ]:
import os
import re
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForQuestionAnswering

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

# 어떻게 디벨롭 가능할까??????????????????? 모르겠음...크리스마스는 솔직히..답변 불가능한거 아니냐

print("Loading BLIP-VQA model...")
model_name = "Salesforce/blip-vqa-capfilt-large"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForQuestionAnswering.from_pretrained(model_name).to(device)
model.eval()
print("BLIP-VQA model loaded!")

def validate_image_path(img_path):
    if img_path is None or not isinstance(img_path, str) or img_path.strip() == "":
        return None
    colab_path = img_path.replace("./input_images/", "/content/input_images/")
    if os.path.exists(colab_path):
        return colab_path
    for path in [
        img_path,
        os.path.join("/content", img_path.lstrip("./")),
        os.path.join("/content/input_images", os.path.basename(img_path)),
        os.path.join("/content/drive/MyDrive/input_images", os.path.basename(img_path))
    ]:
        if os.path.exists(path):
            return path
    return None

def extract_answer_letter(text):
    text = text.upper()
    candidates = re.findall(r'\b[ABCD]\b', text)
    if candidates:
        return candidates[0]
    for ch in ['A', 'B', 'C', 'D']:
        if ch in text:
            return ch
    return "?"

def generate_answer(image, question, choices):
    # BLIP-VQA는 이미지+질문 입력만 받음
    # choices는 prompt에 포함시켜 질문을 더 명확히 할 수 있지만 모델에 직접 선택지 입력 안 됨
    prompt = question.strip()
    inputs = processor(image, prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        answer = processor.tokenizer.decode(outputs.logits.argmax(-1), skip_special_tokens=True).strip()

    # 답변이 선택지 중 하나의 글자인지 추출
    answer_letter = extract_answer_letter(answer)
    if answer_letter == "?":
        # 선택지와 무관한 텍스트 답변일 경우, 선택지 중 답과 가장 유사한 걸 찾는 등 추가 로직 필요할 수 있음
        # 여기선 간단히 "?" 처리
        answer_letter = "?"

    return answer_letter, answer

# 데이터 로드
data_path = "/content/dev_test.csv"
for alt_path in [
    data_path,
    "./dev_test.csv",
    "/content/drive/MyDrive/dev_test.csv",
    "./dataset/dev_test.csv"
]:
    if os.path.exists(alt_path):
        data_path = alt_path
        break
else:
    raise FileNotFoundError("CSV 파일을 찾을 수 없습니다.")

print(f"Loading data from: {data_path}")
test_df = pd.read_csv(data_path)
print(f"Loaded {len(test_df)} samples")

results, logs = [], []

print("Starting inference...")
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    try:
        img_path = validate_image_path(row.get("img_path", None))
        question = row["Question"]
        choices = [row["A"], row["B"], row["C"], row["D"]]

        if img_path is None:
            # 이미지 없으면 규칙상 답변 불가 처리
            results.append("?")
            logs.append("No image provided")
            continue

        image = Image.open(img_path).convert("RGB")

        answer, gen = generate_answer(image, question, choices)

        results.append(answer)
        logs.append(gen)

        if idx < 5 or idx % 20 == 0:
            print(f"[{idx}] Question: {question}")
            print(f"[{idx}] Answer: {answer}")
            print(f"[{idx}] Model Output: {gen}")
            print("-" * 50)

    except Exception as e:
        results.append("?")
        logs.append(f"Error: {e}")
        print(f"[ERROR {idx}] {e}")

output_dir = "/content/output"
os.makedirs(output_dir, exist_ok=True)

submission = pd.DataFrame({"ID": test_df["ID"], "answer": results})
submission.to_csv(os.path.join(output_dir, "final_submit.csv"), index=False)
submission.to_csv("/content/final_submit.csv", index=False)

pd.DataFrame({
    "ID": test_df["ID"],
    "Question": test_df["Question"],
    "answer": results,
    "generated_text": logs
}).to_csv("/content/debug_results.csv", index=False)

print("Inference completed!")